# Setup R & Model Dependencies (Kaggle & Colab Compatible)

This notebook sets up the environment and installs all necessary system, Python `rpy2`, `m2cgen`, `scikit-learn`, and R packages required to train Random Forest, SVM, One-vs-Rest Logistic Regression, Hierarchical Extreme Models, XGBoost / LightGBM, **Kernel Density Estimation (KDE) Synthetic Data Generators (`ks`, `MASS`)**, and **Hyperparameter Tuning Frameworks (`mlr3`, `mlr3learners`, `mlr3tuning`, `bbotk`, `paradox`)** on `.RData` datasets, both locally and in **Kaggle** / **Google Colab** environments.

### Dependencies Installed:
- **Python Libraries**: `rpy2`, `m2cgen`, `scikit-learn` (enables `%%R` magic, model transpilation to C, and scikit-learn model export)
- **R Base & Dev Tools**: `r-base`, `r-base-dev`
- **Machine Learning & Boosting Libraries**: `randomForest`, `ranger`, `caret`, `e1071`, `nnet`, `xgboost`, `lightgbm`
- **Synthetic Sample Generation & Density Estimation**: `ks`, `MASS`
- **Hyperparameter Tuning & Optimization Framework**: `mlr3`, `mlr3learners`, `mlr3tuning`, `bbotk`, `paradox`
- **Data Processing & Utilities**: `jsonlite`, `dplyr`, `data.table`
- **Visualization & Metrics**: `ggplot2`, `pROC`
- **Jupyter R Kernel**: `IRkernel`

In [ ]:
%%bash
# Step 1: Detect environment and install R base system packages safely
export DEBIAN_FRONTEND=noninteractive

SUDO=""
if [ "$(id -u)" -ne 0 ] && command -v sudo >/dev/null 2>&1; then
    SUDO="sudo"
fi

echo "===> Checking R installation..."
if command -v Rscript >/dev/null 2>&1; then
    echo "R is already installed:"
    Rscript --version || true
else
    echo "R not found. Installing system dependencies..."
    $SUDO apt-get update -qq || true
    $SUDO apt-get install -y --no-install-recommends \
        software-properties-common dirmngr wget curl ca-certificates build-essential \
        libcurl4-openssl-dev libssl-dev libxml2-dev || true

    echo "===> Adding CRAN repository..."
    wget -qO- https://cloud.r-project.org/bin/linux/ubuntu/marutter_pubkey.asc | $SUDO tee /etc/apt/trusted.gpg.d/cran_ubuntu_key.asc > /dev/null 2>&1 || true
    $SUDO add-apt-repository -y "deb https://cloud.r-project.org/bin/linux/ubuntu $(lsb_release -cs 2>/dev/null || echo 'jammy')-cran40/" || true
    $SUDO apt-get update -qq || true
    $SUDO apt-get install -y --no-install-recommends r-base r-base-dev || true
fi

In [ ]:
%%bash
# Step 2: Install fast prebuilt R binary packages via APT & Python dependencies
export DEBIAN_FRONTEND=noninteractive

SUDO=""
if [ "$(id -u)" -ne 0 ] && command -v sudo >/dev/null 2>&1; then
    SUDO="sudo"
fi

echo "===> Ensuring Python rpy2, m2cgen, and scikit-learn are installed..."
pip install --quiet rpy2 m2cgen scikit-learn || true

echo "===> Installing R prebuilt binary packages via APT..."
$SUDO apt-get update -qq || true
$SUDO apt-get install -y --no-install-recommends \
    r-cran-randomforest \
    r-cran-ranger \
    r-cran-caret \
    r-cran-e1071 \
    r-cran-nnet \
    r-cran-jsonlite \
    r-cran-dplyr \
    r-cran-ggplot2 \
    r-cran-proc \
    r-cran-data.table \
    r-cran-xgboost \
    r-cran-lightgbm \
    r-cran-ks \
    r-cran-mass \
    r-cran-mlr3 \
    r-cran-mlr3tuning \
    r-cran-bbotk \
    r-cran-paradox \
    r-cran-irkernel \
    r-cran-gridextra || true

In [ ]:
%%bash
# Step 3: Verify and install missing packages via CRAN & register Jupyter IRkernel
Rscript -e '
required_pkgs <- c("randomForest", "ranger", "caret", "e1071", "nnet", "jsonlite", "dplyr", "ggplot2", "pROC", "data.table", "xgboost", "lightgbm", "ks", "MASS", "mlr3", "mlr3learners", "mlr3tuning", "bbotk", "paradox", "IRkernel", "gridExtra")
missing_pkgs <- required_pkgs[!(required_pkgs %in% installed.packages()[,"Package"])]

if (length(missing_pkgs) > 0) {
  cat("Installing missing R packages from CRAN:", paste(missing_pkgs, collapse=", "), "\n")
  try(install.packages(missing_pkgs, repos="https://cloud.r-project.org", Ncpus = parallel::detectCores()))
} else {
  cat("All required R packages are already installed!\n")
}
# Register IRkernel for Jupyter / Kaggle
if (requireNamespace("IRkernel", quietly = TRUE)) {
  cat("Registering IRkernel for Jupyter notebook...\n")
  tryCatch({
    IRkernel::installspec(user = FALSE)
  }, error = function(e) {
    try(IRkernel::installspec(user = TRUE))
  })
}
'

In [ ]:
%%bash
# Step 4: Verification check of R environment and installed packages
echo "===> Verification Report:"
Rscript -e '
cat("R Version:", R.version.string, "\n\n")
cat(sprintf("%-18s %-12s\n", "Package", "Status"))
cat(paste(rep("-", 30), collapse=""), "\n")
pkgs <- c("randomForest", "ranger", "caret", "e1071", "nnet", "jsonlite", "dplyr", "ggplot2", "pROC", "data.table", "xgboost", "lightgbm", "ks", "MASS", "mlr3", "mlr3learners", "mlr3tuning", "bbotk", "paradox", "IRkernel")
all_ok <- TRUE
for (p in pkgs) {
  avail <- requireNamespace(p, quietly = TRUE)
  if (!avail) all_ok <- FALSE
  cat(sprintf("%-18s %-12s\n", p, ifelse(avail, "[OK]", "[MISSING]")))
}
cat(paste(rep("-", 30), collapse=""), "\n")
if (all_ok) {
  cat("SUCCESS: All R dependencies are ready for training models!\n")
} else {
  cat("NOTE: Some optional packages are missing. Re-run setup if needed.\n")
}
'

## Project Pipeline Notebooks
The project consists of the following modular notebooks in `models/`:
1. **`models/train_hierarchical_lightgbm_layers.ipynb`**: Trains Layer 1 LightGBM (ESI 1 Detector on complete data) & Layer 2 LightGBM (ESI 2/3 vs 4/5 Specialist without ESI 1 rows).
2. **`models/rf_esi23_esi45_extreme.ipynb`**: Microcontroller-Optimized Layer 2 Model (`"2_3"`, `"4_5"`, `"1"`).
3. **`models/lightgbm_esi23.ipynb`**: Layer 3A LightGBM Specialist (ESI 2 vs ESI 3).
4. **`models/lightgbm_esi45.ipynb`**: Layer 3B LightGBM Specialist (ESI 4 vs ESI 5).
5. **`models/combined_hierarchical_triage_pipeline.ipynb`**: Master 3-Layer Pipeline Benchmark (Soft Probabilistic Joint vs Hard Case-Selector).
6. **`models/transpile_soft_pipeline_to_c.ipynb`**: Native C Transpilation & Verification via `m2cgen` (< 200 KB C code footprint).
7. **`models/lof_esi1_anomaly_detection.ipynb`**: ESP32 Microcontroller Centroid Local Outlier Factor (LOF) Anomaly Detector for ESI 1 (~28 KB C Flash array).
8. **`models/autoencoder_esi1_anomaly_detection.ipynb`**: ESP32 Microcontroller Autoencoder Anomaly Detector for ESI 1 (~6.8 KB C Flash weight matrix).